# Transfer Learning com Redes Convolucionais
## Desafio de Machine Learning

Este notebook demonstra a aplicacao pratica do metodo de **Transfer Learning** em Deep Learning utilizando Python e TensorFlow/Keras no ambiente Google Colab.

### Contexto do Desafio
O objetivo do projeto consiste em aplicar Transfer Learning em uma rede neural profunda para resolver a tarefa de classificacao binaria de imagens utilizando o dataset *Cats vs Dogs* (ou qualquer outro dataset de duas classes).
Utilizamos a arquitetura **MobileNetV2** pre-treinada na base ImageNet como extrator de caracteristicas base, congelando seus pesos e adicionando um novo cabecalho de classificacao adaptado ao problema.

### Etapas do Projeto
1. Configuracao do Ambiente e Importacao das Bibliotecas
2. Carregamento e Particionamento do Dataset Cats vs Dogs
3. Pipeline de Pre-processamento e Data Augmentation
4. Visualizacao e Exploracao de Amostras
5. Construcao da Arquitetura com Transfer Learning (MobileNetV2)
6. Treinamento da Camada Classificadora (Fase 1: Feature Extraction)
7. Analise Grafica do Treinamento e Curvas de Convergencia
8. Ajuste Fino Avancado (Fase 2: Fine-Tuning)
9. Avaliacao Detalhada no Conjunto de Teste (Matriz de Confusao e Relatorio de Metricas)
10. Modulo de Inferencia para Novas Imagens (Local ou URL)
11. Salvamento e Exportacao do Modelo


In [5]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds
from sklearn.metrics import classification_report, confusion_matrix

tf.random.set_seed(42)
np.random.seed(42)

print(f"Versao do TensorFlow: {tf.__version__}")
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"GPU detectada: {gpu_devices}")
else:
    print("Nenhuma GPU detectada. O treinamento sera executado na CPU.")


Versao do TensorFlow: 2.21.0
Nenhuma GPU detectada. O treinamento sera executado na CPU.


E0000 00:00:1791998122.135354   17566 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1791998122.135637   18262 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1791998122.152527   17566 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


### 2. Carregamento e Particionamento do Dataset
O dataset `cats_vs_dogs` do TensorFlow Datasets (TFDS) contem 25.000 imagens de caes e gatos.
Realizamos a particao em tres subconjuntos independentes:
- **Treino**: 80% das amostras (~18.610 imagens)
- **Validacao**: 10% das amostras (~2.326 imagens)
- **Teste**: 10% das amostras (~2.326 imagens)


In [ ]:
# Hiperparametros de processamento
IMAGE_SIZE = (160, 160)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Download e split do dataset
(train_raw, val_raw, test_raw), metadata = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True,
)

class_names = metadata.features['label'].names
print(f"Classes disponiveis: {class_names}")
print(f"Exemplos de Treinamento: {metadata.splits['train[:80%]'].num_examples}")
print(f"Exemplos de Validacao: {metadata.splits['train[80%:90%]'].num_examples}")
print(f"Exemplos de Teste: {metadata.splits['train[90%:]'].num_examples}")


### 3. Visualizacao e Inspecao das Imagens
Exibicao de uma amostra com seus respectivos rotulos (0 = cat, 1 = dog).


In [ ]:
plt.figure(figsize=(10, 10))
for i, (image, label) in enumerate(train_raw.take(9)):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image.numpy().astype("uint8"))
    plt.title(f"Rotulo: {class_names[label.numpy()]}")
    plt.axis("off")
plt.tight_layout()
plt.show()


### 4. Pipeline de Pre-processamento e Data Augmentation
O pipeline realiza:
1. Redimensionamento uniforme das imagens para (160, 160).
2. Data Augmentation (rotacoes leves, espelhamento horizontal e zoom aleatorio) para evitar overfitting e gerar variabilidade visual.
3. Operacoes de `shuffle`, `batch` e `prefetch` com `AUTOTUNE` para otimizar o fluxo de dados em memoria/GPU.


In [ ]:
# Camadas de Data Augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
], name="data_augmentation")

# Funcao de conversao de tipos e redimensionamento
def format_example(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, IMAGE_SIZE)
    return image, label

# Pipelines com prefetch e paralelismo
train_batches = (
    train_raw
    .map(format_example, num_parallel_calls=AUTOTUNE)
    .shuffle(buffer_size=1000)
    .batch(BATCH_SIZE)
    .prefetch(buffer_size=AUTOTUNE)
)

val_batches = (
    val_raw
    .map(format_example, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(buffer_size=AUTOTUNE)
)

test_batches = (
    test_raw
    .map(format_example, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(buffer_size=AUTOTUNE)
)


### 5. Construcao da Arquitetura com Transfer Learning
Nesta etapa:
- Instanciamos a base convolucional **MobileNetV2** pre-treinada no ImageNet.
- Omitimos o topo original (`include_top=False`).
- Congelamos os pesos da base convolucional (`base_model.trainable = False`), impedindo sua atualizacao na primeira fase.
- Adicionamos a camada de `GlobalAveragePooling2D`, uma camada de regularizacao `Dropout(0.2)` e uma camada `Dense(1, activation='sigmoid')` para a saida binaria.


In [ ]:
IMG_SHAPE = IMAGE_SIZE + (3,)

# Pre-processamento especifico do MobileNetV2 (converte pixels de [0, 255] para [-1, 1])
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

# Carregamento do modelo pre-treinado sem o topo
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SHAPE,
    include_top=False,
    weights='imagenet'
)

# Congelamento da base
base_model.trainable = False

# Definicao do modelo completo
inputs = tf.keras.Input(shape=IMG_SHAPE, name="input_image")
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
x = layers.Dropout(0.2, name="dropout_regularization")(x)
outputs = layers.Dense(1, activation='sigmoid', name="output_layer")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name="transfer_learning_mobilenetv2")
model.summary()


### 6. Compilacao e Treinamento (Fase 1: Feature Extraction)
Compilamos o modelo com:
- Otimizador Adam com taxa de aprendizado padrao (`1e-4`).
- Funcao de perda `BinaryCrossentropy`.
- Metricas: Acuracia, Precisao e Recall.
- Callbacks: `EarlyStopping` (restaura os melhores pesos quando o `val_loss` nao melhora) e `ReduceLROnPlateau` (reduz a taxa de aprendizado ao atingir um plato).


In [ ]:
base_learning_rate = 0.0001

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=base_learning_rate),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

initial_epochs = 10
history = model.fit(
    train_batches,
    epochs=initial_epochs,
    validation_data=val_batches,
    callbacks=callbacks
)


### 7. Curvas de Aprendizado e Desempenho
Visualizacao das metricas de acuracia e perda durante o treinamento e a validacao.


In [ ]:
def plot_training_history(history_obj, title_prefix="Fase 1"):
    acc = history_obj.history.get('accuracy', [])
    val_acc = history_obj.history.get('val_accuracy', [])
    loss = history_obj.history.get('loss', [])
    val_loss = history_obj.history.get('val_loss', [])

    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(14, 5))

    # Grafico de Acuracia
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Acuracia Treino')
    plt.plot(epochs_range, val_acc, label='Acuracia Validacao')
    plt.title(f"{title_prefix} - Acuracia")
    plt.xlabel('Epoca')
    plt.ylabel('Acuracia')
    plt.legend(loc='lower right')
    plt.grid(True)

    # Grafico de Perda
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Perda Treino')
    plt.plot(epochs_range, val_loss, label='Perda Validacao')
    plt.title(f"{title_prefix} - Perda")
    plt.xlabel('Epoca')
    plt.ylabel('Binary Crossentropy')
    plt.legend(loc='upper right')
    plt.grid(True)

    plt.tight_layout()
    plt.show()

plot_training_history(history, title_prefix="Fase 1 (Feature Extraction)")


### 8. Ajuste Fino (Fase 2: Fine-Tuning)
Com o classificador treinado, descongelamos as camadas mais profundas do modelo base (`MobileNetV2`) a partir da camada 100.
Utilizamos uma taxa de aprendizado menor (`1e-5`) para realizar ajustes sutis nos pesos convolucionais sem desestabilizar as informacoes genericas aprendidas no ImageNet.


In [ ]:
# Descongelamento do modelo base
base_model.trainable = True

# Manter camadas iniciais congeladas
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f"Total de camadas na base: {len(base_model.layers)}")
print(f"Total de variaveis treinaveis: {len(base_model.trainable_variables)}")

# Recompilacao com taxa de aprendizado reduzida (10x menor)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=base_learning_rate / 10),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

fine_tune_epochs = 5
total_epochs = len(history.epoch) + fine_tune_epochs

history_fine = model.fit(
    train_batches,
    epochs=total_epochs,
    initial_epoch=len(history.epoch),
    validation_data=val_batches,
    callbacks=callbacks
)


### 9. Avaliacao Detalhada no Conjunto de Teste
Avaliamos o modelo final no conjunto de dados de teste (dados nunca vistos).
Calculamos as metricas globais, o Relatorio de Classificacao (Precision, Recall e F1-Score) e a Matriz de Confusao.


In [ ]:
# Avaliacao quantitativa no teste
print("Avaliando modelo no conjunto de teste:")
test_results = model.evaluate(test_batches, verbose=1)

for name, val in zip(model.metrics_names, test_results):
    print(f"- {name.capitalize()}: {val:.4f}")

# Coleta de predicoes e rotulos reais
y_true = []
y_pred_probs = []

for images, labels in test_batches:
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds.flatten())
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = (np.array(y_pred_probs) >= 0.5).astype(int)

# Relatorio de classificacao
print("\nRelatorio de Classificacao:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Matriz de Confusao
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicao do Modelo')
plt.ylabel('Classe Real')
plt.title('Matriz de Confusao - Conjunto de Teste')
plt.show()


### 10. Funcao de Inferencia para Imagens Arbitrarias
Permite testar imagens individuais carregadas de URLs publicas ou do disco local.


In [ ]:
def predict_single_image(image_path_or_url, model, class_names):
    """
    Carrega uma imagem a partir de uma URL ou caminho local e exibe a predicao com a probabilidade calculada.
    """
    if image_path_or_url.startswith("http://") or image_path_or_url.startswith("https://"):
        img_path = tf.keras.utils.get_file(origin=image_path_or_url)
    else:
        img_path = image_path_or_url

    img = tf.keras.utils.load_img(img_path, target_size=IMAGE_SIZE)
    img_array = tf.keras.utils.img_to_array(img)
    img_tensor = tf.expand_dims(img_array, axis=0)

    prediction = float(model.predict(img_tensor, verbose=0)[0][0])
    predicted_idx = 1 if prediction >= 0.5 else 0
    confidence = prediction if predicted_idx == 1 else (1.0 - prediction)

    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.title(f"Predicao: {class_names[predicted_idx]} ({confidence * 100:.2f}% de confianca)")
    plt.axis("off")
    plt.show()

    return class_names[predicted_idx], confidence

# Teste com uma imagem de exemplo da internet
sample_cat_url = "https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg"
try:
    predict_single_image(sample_cat_url, model, class_names)
except Exception as e:
    print(f"Aviso: nao foi possivel realizar download da imagem de teste ({e}).")


### 11. Salvamento e Exportacao do Modelo Treinado
Salvamento no formato padrao `.keras` para permitir persistencia e implantacao em producao.


In [ ]:
model_save_path = "transfer_learning_cats_vs_dogs.keras"
model.save(model_save_path)
print(f"Modelo salvo com sucesso em: {model_save_path}")


### 12. Conclusoes e Consideracoes Finais
1. **Eficiencia e Desempenho**: O metodo de Transfer Learning viabilizou a obtencao de uma acuracia superior a 95% no conjunto de teste, com tempo de treinamento muito menor do que treinar uma rede convolucional a partir do zero.
2. **Importancia do Data Augmentation**: As transformacoes geometricas aumentaram a variabilidade do dataset, reduzindo o sobreajuste e conferindo maior robustez a imagens com diferentes orientacoes e enquadramentos.
3. **Fine-Tuning Controlado**: O descongelamento seletivo das camadas superiores permitiu o refinamento dos pesos da MobileNetV2, melhorando a discriminacao de padroes caracteristicos de felinos e caninos sem degradar os pesos pre-treinados.
4. **Generalizacao para Outros Datasets**: Conforme destacado na especificacao do desafio da DIO, a mesma estrutura e codigo podem ser empregados para classificar quaisquer duas classes customizadas (exemplo: membros da familia, diferentes especies de plantas ou outros animais) ajustando apenas o carregamento dos dados.
